# 01 — Policy Evaluation
**Week 4 | Dynamic Programming**

**Policy Evaluation** answers: *How good is a given policy π?*

We iteratively apply the Bellman expectation equation until V converges:

$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V_k(s') \right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# ---- Reuse Grid World from Week 3 ----
class GridWorld:
    ACTIONS = {0:(-1,0), 1:(1,0), 2:(0,-1), 3:(0,1)}
    ACTION_SYMBOLS = {0:'↑', 1:'↓', 2:'←', 3:'→'}
    def __init__(self, size=5):
        self.size = size; self.start=(0,0); self.goal=(size-1,size-1)
        self.pits = {(1,1),(1,3),(3,1),(3,3)}
    def n_states(self): return self.size**2
    def n_actions(self): return 4
    def transitions(self, s, a):
        """Return list of (prob, next_s, reward, done)."""
        r, c = divmod(s, self.size)
        dr, dc = self.ACTIONS[a]
        nr = max(0, min(self.size-1, r+dr)); nc = max(0, min(self.size-1, c+dc))
        ns = nr*self.size + nc
        if (nr,nc) == self.goal:       return [(1.0, ns, +10.0, True)]
        if (nr,nc) in self.pits:       return [(1.0, ns,  -5.0, True)]
        return [(1.0, ns, -0.1, False)]

env = GridWorld()

In [ ]:
def policy_evaluation(env, policy, gamma=0.99, theta=1e-6, max_iter=1000):
    """
    Iterative policy evaluation.
    policy: array of shape (n_states, n_actions) — action probabilities.
    Returns V: array of shape (n_states,)
    """
    V = np.zeros(env.n_states())
    history = [V.copy()]
    for iteration in range(max_iter):
        delta = 0.0
        V_new = np.zeros_like(V)
        for s in range(env.n_states()):
            v = 0.0
            for a in range(env.n_actions()):
                for prob, ns, reward, done in env.transitions(s, a):
                    v += policy[s, a] * prob * (reward + (0 if done else gamma * V[s]))
            V_new[s] = v
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        history.append(V.copy())
        if delta < theta:
            print(f"Converged in {iteration+1} iterations (Δ={delta:.2e})")
            break
    return V, history

In [ ]:
# Uniform random policy
n_s, n_a = env.n_states(), env.n_actions()
uniform_policy = np.ones((n_s, n_a)) / n_a

V, history = policy_evaluation(env, uniform_policy)

In [ ]:
# Visualise final value function
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Heatmap
im = axes[0].imshow(V.reshape(5,5), cmap='RdYlGn')
plt.colorbar(im, ax=axes[0])
for r in range(5):
    for c in range(5):
        label = ''
        if (r,c)==env.goal: label='G'
        elif (r,c)==env.start: label='S'
        elif (r,c) in env.pits: label='✕'
        axes[0].text(c, r, f'{label}\n{V[r*5+c]:.1f}', ha='center', va='center', fontsize=9)
axes[0].set_title('V^π (uniform random policy)'); axes[0].set_xticks([]); axes[0].set_yticks([])

# Convergence curve
max_delta = [np.max(np.abs(history[i+1]-history[i])) for i in range(len(history)-1)]
axes[1].semilogy(max_delta, color='steelblue', linewidth=2)
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Max |ΔV|')
axes[1].set_title('Convergence of Policy Evaluation')
plt.tight_layout(); plt.show()

## ✅ Exercises
1. Try γ=0.5, γ=0.9, γ=0.999. How does the value function shape change?
2. Create a policy that always moves RIGHT. Evaluate it. What happens to states on the rightmost column?
3. **Challenge**: implement **in-place** policy evaluation (update V[s] immediately instead of V_new). Does it converge faster?

Solution 1-

$\gamma=0.5$: The values decay extremely rapidly as you move away from the goal. Only the cells directly adjacent to the goal retain noticeable positive value, resulting in a very steep drop-off.$\gamma=0.9$: This is a balanced value landscape. States further away still possess reasonable values because the agent expects to retain 90% of downstream rewards per step.$\gamma=0.999$: The value landscape becomes exceptionally flat across the grid. Because the step penalty is minor compared to a near-undiscounted future goal reward, almost all empty states converge to values close to the ultimate goal amount.

Solution 2-

Under a deterministic policy that forces the agent RIGHT (Action 3):

Cells to the left of the goal or pits will stream straight into them, accumulating predictable deterministic terminal returns.

For states located in the rightmost column, attempting to move right causes the agent to hit the boundary grid wall and bounce back into the exact same state. As a result, these states enter an infinite loop of accumulating the negative living step cost (e.g., $-0.1$). Their values will continuously decrease and diverge towards negative infinity ($-\infty$), or reach a depressed limit depending on your terminal setups.

Solution 3-

Yes, it converges significantly faster. Standard synchronous evaluation requires two separate memory arrays ($V_k$ and $V_{k+1}$), meaning updates do not propagate to neighboring states until the next full sweep. In-place evaluation overwrites values immediately, allowing subsequent state calculations within the same iteration loop to use the updated, more accurate neighboring values. This sweeps information through the grid faster and roughly cuts the number of required iterations in half.

In [ ]:
def policy_evaluation_inplace(env, policy, gamma=0.99, theta=1e-6):
    V = np.zeros(env.n_states())
    while True:
        delta = 0
        for s in range(env.n_states()):
            v_old = V[s]
            v_new = 0
            for a in range(env.n_actions()):
                prob_a = policy[s, a]
                # In-place uses current V directly
                next_s, r, term = env.transitions(s, a)
                v_new += prob_a * (r + (0.0 if term else gamma * V[next_s]))
            V[s] = v_new
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:
            break
    return V